<a href="https://colab.research.google.com/github/tsilva/aiml-notebooks/blob/main/rnn/wip_rnn_variable_length_bit_flip.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RNN - Variable Length Bit Flipper

In this notebook we will train a RNN to learn how to flip bits in sequences of arbritary length.

## Setup

In [3]:
!pip install tsilva-notebook-utils==0.0.13

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 25.2 MB/s eta 0:00:00


Load secrets:

In [4]:
from tsilva_notebook_utils.colab import load_secrets_into_env
load_secrets_into_env([
    'HF_TOKEN',
    'WANDB_API_KEY',
    'NOTIFICATION_URL',
    'NOTIFICATION_AUTH_TOKEN'
])

Define configuration:

In [5]:
import os
from tsilva_notebook_utils.colab import notebook_id_from_title

def setup_config():
    # @markdown ### 🌱 Reproducibility Settings

    # @markdown Random seed for reproducibility
    seed = 42  # @param {type:"integer"}

    # @markdown ### 🧩 Dataset Settings

    # @markdown Total size of the synthetic dataset
    dataset_size = 10_000  # @param {type:"integer"}

    # @markdown Minimum length of input sequence (time steps)
    min_seq_length = 2  # @param {type:"integer"}

    # @markdown Maximum length of input sequence (time steps)
    max_seq_length = 50  # @param {type:"integer"}

    # @markdown ### 🏋️ Training Settings

    # @markdown Number of training epochs
    n_epochs = 5  # @param {type:"integer"}

    # @markdown Batch size for training
    batch_size = 256  # @param {type:"integer"}

    # @markdown Learning rate for the optimizer
    learning_rate = 0.001  # @param {type:"number"}

    # @markdown ### 🧠 Model Architecture Settings

    # @markdown Input size (number of input features)
    input_size = 1  # @param {type:"integer"}

    # @markdown Hidden layer size
    hidden_size = 16  # @param {type:"integer"}

    # @markdown Output size (number of output features)
    output_size = 1  # @param {type:"integer"}

    # Generate notebook id from notebook title
    os.environ["NOTEBOOK_ID"] = notebook_id_from_title()

    return {
        'seed': seed,
        'n_epochs': n_epochs,
        'batch_size': batch_size,
        'dataset_size': dataset_size,
        'min_seq_length': min_seq_length,
        'max_seq_length': max_seq_length,
        'learning_rate': learning_rate,
        'input_size': input_size,
        'hidden_size': hidden_size,
        'output_size': output_size
    }

CONFIG = setup_config()

Let's get started and install the necessary packages: 🛠️

In [6]:
!pip install wandb

First let's build a dataset:

In [7]:
import random
import torch
from torch.utils.data import Dataset, DataLoader

# Define a custom dataset class for the bit-flip task
class BitFlipDataset(Dataset):
    def __init__(
        self,
        dataset_size=None,
        min_seq_length=None,
        max_seq_length=None
    ):
        # Use default values from CONFIG if arguments are not provided
        if dataset_size is None: dataset_size = CONFIG['dataset_size']
        if min_seq_length is None: min_seq_length = CONFIG['min_seq_length']
        if max_seq_length is None: max_seq_length = CONFIG['max_seq_length']

        self.data = []  # Initialize an empty list to store data samples

        # Generate the dataset
        for _ in range(dataset_size):
            # Randomly select a sequence length within the given range
            seq_len = random.randint(min_seq_length, max_seq_length)

            # Generate a random binary sequence (0s and 1s), shape: (seq_len, 1)
            X = torch.randint(0, 2, (seq_len, 1)).float()

            # Create the target sequence by flipping the bits (1 -> 0, 0 -> 1)
            Y = 1.0 - X

            # Append the input-output pair to the dataset
            self.data.append((X, Y))

    # Return the total number of samples in the dataset
    def __len__(self):
        return len(self.data)

    # Retrieve a sample by index
    def __getitem__(self, idx):
        return self.data[idx]

train_dataset = BitFlipDataset()
train_dataset[0]

(tensor([[0.],
         [1.],
         [0.],
         [1.],
         [0.],
         [1.],
         [0.],
         [1.],
         [1.],
         [0.],
         [1.],
         [1.],
         [0.],
         [0.],
         [0.],
         [0.],
         [0.],
         [1.],
         [1.],
         [0.],
         [0.],
         [0.],
         [1.],
         [1.],
         [0.],
         [0.],
         [1.],
         [0.],
         [1.],
         [1.],
         [0.],
         [1.],
         [0.],
         [0.],
         [1.],
         [0.],
         [1.],
         [0.],
         [1.],
         [1.],
         [1.],
         [0.],
         [0.],
         [1.],
         [0.],
         [1.],
         [1.]]),
 tensor([[1.],
         [0.],
         [1.],
         [0.],
         [1.],
         [0.],
         [1.],
         [0.],
         [0.],
         [1.],
         [0.],
         [0.],
         [1.],
         [1.],
         [1.],
         [1.],
         [1.],
         [0.],
         [0.],
        

Now let's build a data loader for the dataset. To use a data loader on samples of different length we'll need to provide a custom collator. By default pytorch will try to stack the tensors when loading a batch, but you can't stack vectors of different length, therefore we need to pad them to max batch length:

In [8]:
import torch.nn as nn

def collate_fn(batch):
    seqs_x, seqs_y = zip(*batch)
    padded_x = nn.utils.rnn.pad_sequence(seqs_x, batch_first=True, padding_value=-1)
    padded_y = nn.utils.rnn.pad_sequence(seqs_y, batch_first=True, padding_value=-1)
    return padded_x, padded_y

collate_fn([
    (torch.tensor([2, 2, 2, 2], dtype=torch.long), torch.tensor([3, 3, 3, 3], dtype=torch.long)),
    (torch.tensor([1, 1], dtype=torch.long), torch.tensor([2, 2], dtype=torch.long)),
    (torch.tensor([3, 3, 3], dtype=torch.long), torch.tensor([4, 4, 4], dtype=torch.long))
])

(tensor([[ 2,  2,  2,  2],
         [ 1,  1, -1, -1],
         [ 3,  3,  3, -1]]),
 tensor([[ 3,  3,  3,  3],
         [ 2,  2, -1, -1],
         [ 4,  4,  4, -1]]))

Test the data loader with collate function:

In [9]:
batch_size = 10
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
x, y = next(iter(train_loader))
x.shape

torch.Size([10, 50, 1])

Create the model:

In [10]:
class SimpleRNN(nn.Module):
    def __init__(
        self,
        input_size=None,
        hidden_size=None,
        output_size=None
    ):
        super().__init__()

        # Use default values from CONFIG if not provided
        if input_size is None: input_size = CONFIG['input_size']
        if hidden_size is None: hidden_size = CONFIG['hidden_size']
        if output_size is None: output_size = CONFIG['output_size']

        # Define an RNN layer
        self.rnn = nn.RNN(input_size, hidden_size, batch_first=True)

        # Fully connected layer to map hidden state output to desired output size
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x, return_hidden=False):
        # Pass the input through the RNN layer
        out, _ = self.rnn(x)  # out: (batch, seq_len, hidden_size)

        # Pass the RNN output through the fully connected layer
        output = self.fc(out)  # output: (batch, seq_len, output_size)

        if return_hidden:
            return output, out  # out is the hidden states
        else:
            return output

model = SimpleRNN()

Login to wandb and show dashboard:

In [11]:
from tsilva_notebook_utils.wandb import init_with_defaults
init_with_defaults(CONFIG)

wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: tsilva to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


Train the model:

In [12]:
import torch
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import random
import wandb

# Optionally: Watch the model to log gradients and model topology
wandb.watch(model, log="all")

# Define the loss function as Mean Squared Error loss
loss_fn = nn.MSELoss()

# Set learning rate from configuration
learning_rate = CONFIG['learning_rate']

# Initialize the Adam optimizer with model parameters and the learning rate
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# Get the number of training epochs from configuration
n_epochs = CONFIG['n_epochs']

# Training loop with progress bar using tqdm
with tqdm(range(n_epochs), desc="Training") as pbar:
    for epoch in pbar:
        total_loss = 0  # Initialize total loss for the epoch

        # Iterate over batches of data from the training data loader
        for x_batch, y_batch in train_loader:
            # Forward pass: compute model predictions and capture hidden states
            outputs, hidden_states = model(x_batch, return_hidden=True)

            # Create a mask to ignore padded values (assuming -1 is used for padding)
            mask = (x_batch != -1).float()

            # Compute the loss, applying the mask to ignore padded elements
            loss = loss_fn(outputs * mask, y_batch * mask)

            # Backpropagation step
            optimizer.zero_grad()  # Clear previous gradients
            loss.backward()        # Compute gradients
            optimizer.step()       # Update model parameters

            # Accumulate the loss value for this batch
            total_loss += loss.item()

        # Compute average loss for the epoch
        avg_loss = total_loss / len(train_loader)

        # Log average loss to wandb
        wandb.log({'epoch': epoch + 1, 'loss': avg_loss})

        # Update the progress bar with the current epoch and average loss
        pbar.set_postfix({'Epoch': epoch + 1, 'Loss': f'{avg_loss:.6f}'})

wandb.finish()

Training: 100%|██████████| 5/5 [00:42<00:00,  8.43s/it, Epoch=5, Loss=0.000002]


epoch,▁▃▅▆█
loss,█▁▁▁▁
epoch,5
loss,0.0


Test the sequences:

In [13]:
# Generalization Test: Longer sequence
def test_sequence(seq):
    with torch.no_grad():
        seq_tensor = seq.float().unsqueeze(0)  # (1, seq_len, 1)
        output = model(seq_tensor).squeeze(0).squeeze(-1)
        prediction = torch.round(output).int()
        target = (1 - seq.squeeze(-1)).int()

        correct = (prediction == target).sum().item()
        total = target.numel()
        accuracy = correct / total * 100

        print("Input:     ", seq.squeeze(-1).int().tolist())
        print("Prediction:", prediction.tolist())
        print("Target:    ", target.tolist())
        print(f"Accuracy:  {accuracy:.2f}%\n")

# Test: Longer than training
test_seq = torch.tensor([[1], [0]] * 30)  # length = 60
test_sequence(test_seq)

# Test: Shorter than training
test_seq_short = torch.tensor([[1], [0], [1]])  # length = 3
test_sequence(test_seq_short)

# Test: Random length
test_seq_random = torch.randint(0, 2, (17, 1))
test_sequence(test_seq_random)

# Test: Huge length
test_seq = torch.tensor([[1], [0]] * 100)  # length = 200
test_sequence(test_seq)

Input:      [1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0]
Prediction: [0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1]
Target:     [0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1]
Accuracy:  100.00%

Input:      [1, 0, 1]
Prediction: [0, 1, 0]
Target:     [0, 1, 0]
Accuracy:  100.00%

Input:      [1, 0, 1, 1, 1, 1, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 1]
Prediction: [0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1, 1, 1, 0, 0]
Target:     [0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1, 1, 1, 0, 0]
Accuracy:  100.00%

Input:      [1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 

Sending a notification that the training process has completed, allowing asynchronous monitoring of long-running training jobs. Set up automatic runtime disconnection after idle period to save resources:

In [14]:
from tsilva_notebook_utils.colab import notify_and_disconnect_after_timeout
notify_and_disconnect_after_timeout()

Starting idle timeout check. Will disconnect after 300 seconds of no interruption...


Idle Timeout:   1%|▏         | 4/300 [00:04<05:29,  1.11s/s]


KeyboardInterrupt: 